# Chat with your data through a semantic layer

This notebook walks you through answering business questions about the lab's dbt demo shop with the
**[Kubeflow Semantic Operator](https://github.com/kubeflow/semantic-operator)**, from the raw tables to the LLM's answer:

```
 question ──► LLM (Ollama, qwen3:4b) ──MCP──► semantic server ──one governed SQL──► Trino ──► Iceberg tables
                │  picks certified metrics        │  compiles joins, fan-out-safe
                │  and dimensions, never SQL      │  aggregation, row filters, roles
                ◄──────────── rows + the SQL it ran ◄──────────────────────────────┘
```

1. **The data**: seven tables built by dbt, with the traps real data has: orders paid in several parts, refunds in
   installments, costs in a separate table, regions two joins away.
2. **The model**: an [Apache Ossie](https://github.com/apache/ossie) semantic model of that data: datasets,
   relationships, certified metrics, dimensions and roles. The operator publishes it.
3. **The server by hand**: list what it offers, then ask it a question yourself and read the SQL it compiles.
4. **The LLM**: the same question in plain English, step by step: every tool call, its arguments, the server's answer.
5. **Side by side**: six questions, semantic layer vs. the same LLM writing raw SQL, scored against ground truth.
6. **Governance**: a role that may not see e-mails, and a role that only sees its own region.

Runs on the Helm chart with `semantic.enabled=true` (the workstation profile has it). Docker Compose is frozen and has
no semantic layer, so the next cell stops there with a message.

In [ ]:
import json, os, sys, pathlib
import pandas as pd
import yaml
sys.path.append(str(pathlib.Path.cwd().parent))        # lab_utils.py lives one folder up
import lab_utils as lu

lu.require_semantic()                                   # stops on Docker Compose
lu.require_tool_model(lu.TOOL_MODEL)                    # qwen3:4b: Ollama tool calling
MODEL = os.environ.get("SEMANTIC_DEMO_MODEL", "dbt_demo")
print("semantic layer (MCP):", os.environ["SEMANTIC_MCP_URL"])
print("model:", MODEL, "| LLM:", lu.TOOL_MODEL, "at", lu.OLLAMA_URL)

def show(obj, limit=1500):
    """Pretty-print a request or a server answer, cut to `limit` characters."""
    text = json.dumps(obj, indent=2, default=str)
    print(text if len(text) <= limit else text[:limit] + f"\n... ({len(text) - limit} more characters)")

## 1. The data

The dbt project `../dbt/lakehouse_demo` builds the shop from seeds into Trino's `iceberg.dbt_demo` schema:

```
                     dim_regions (country → sales_region)
                           ▲
                     stg_customers ◄──── fct_orders ────► dim_products ────► dim_product_costs
                                          ▲        ▲
                         fct_payments ────┘        └──── fct_refunds
                     (an order can be paid      (an order can be refunded
                      in several parts)          in installments)
```

Build it if it is not there yet (`dbt build`, about a minute). The operator re-checks its model every 30 s and
publishes it once every table exists.

In [ ]:
TABLES = ["fct_orders", "fct_payments", "fct_refunds", "stg_customers", "dim_products", "dim_product_costs", "dim_regions"]
schemas = set(lu.trino("SHOW SCHEMAS FROM iceberg")["Schema"])
present = set(lu.trino("SHOW TABLES FROM iceberg.dbt_demo")["Table"]) if "dbt_demo" in schemas else set()
if not set(TABLES) <= present:
    lu.dbt("build")
pd.DataFrame([(t, int(lu.trino(f"SELECT count(*) FROM iceberg.dbt_demo.{t}").iloc[0, 0])) for t in TABLES],
             columns=["table", "rows"])

What makes this data easy to get wrong, measured directly in SQL:

In [ ]:
lu.trino("""
SELECT
  (SELECT count(*) FROM (SELECT order_id FROM iceberg.dbt_demo.fct_payments GROUP BY 1 HAVING count(*) > 1)) AS orders_paid_in_parts,
  (SELECT count(*) FROM (SELECT order_id FROM iceberg.dbt_demo.fct_refunds  GROUP BY 1 HAVING count(*) > 1)) AS orders_refunded_in_installments,
  (SELECT count(*) FROM iceberg.dbt_demo.fct_orders WHERE status = 'returned')                               AS returned_orders,
  (SELECT count(*) FROM iceberg.dbt_demo.fct_refunds WHERE reason = 'damaged')                               AS goodwill_refunds
""")

## 2. The semantic model

The model is one YAML document (`dbt_demo_model.yaml`, the same file the Helm chart deploys as the SemanticModel
`<release>-data-lab-dbt-demo`). It names the datasets and their keys, how they join, and the **certified metrics**: the
one agreed definition of "revenue", "margin" or "refund rate".

In [ ]:
spec = yaml.safe_load(open("dbt_demo_model.yaml"))
ossie = spec["ossie"]
print(ossie["ai_context"]["instructions"], "\n")
display(pd.DataFrame([{"dataset": d["name"], "primary key": ", ".join(d["primary_key"]), "description": d["description"],
                       "dimensions": ", ".join(f["name"] for f in d["fields"] if "dimension" in f)} for d in ossie["datasets"]]))
display(pd.DataFrame([{"relationship": r["name"], "from": f"{r['from']}.{r['from_columns'][0]}", "to": f"{r['to']}.{r['to_columns'][0]}"}
                      for r in ossie["relationships"]]))
pd.DataFrame([{"metric": m["name"], "definition": m["expression"]["dialects"][0]["expression"], "meaning": m["description"]}
              for m in ossie["metrics"]]).style.set_properties(**{"text-align": "left"})

Who may see what (`governance`): `analyst` sees everything, `marketing` every metric but never the customers'
e-mail, and `dach_sales` only a few metrics and only rows of its own region (a row filter the server adds to every
query).

In [ ]:
pd.DataFrame(spec["governance"]["roles"]).fillna("")

## 3. The semantic server by hand

The server speaks **MCP** (the Model Context Protocol): a small set of tools any agent can call. These are the
calls the LLM will make later; here you make them yourself.

In [ ]:
tools = lu._mcp("tools/list", {}, "analyst")["tools"]
pd.DataFrame([{"tool": t["name"], "what it does": t.get("description", "")} for t in tools])

In [ ]:
show(lu.mcp_call("list_models", {}))
lu.wait_for_model(MODEL)          # the model is served once the operator has checked it against Trino

In [ ]:
print("Certified metrics the analyst role may query:")
display(pd.DataFrame(lu.mcp_call("list_metrics", {"model": MODEL})["metrics"]))
print("Group-by dimensions:")
pd.DataFrame(lu.mcp_call("list_dimensions", {"model": MODEL})["dimensions"])

Now ask a question yourself: *gross margin per product category*. The request names a certified metric and a
dimension; it contains no SQL.

In [ ]:
request = {"model": MODEL, "metrics": ["gross_margin", "total_revenue"], "dimensions": ["dim_products.category"]}
show(request)
answer = lu.mcp_call("query_metric", request)
print("\nSQL the server compiled and ran on Trino:\n")
print(answer["sql"])
pd.DataFrame(answer["rows"], columns=answer["columns"])

Read the SQL: the server found the path `fct_orders → dim_products → dim_product_costs` from the relationships,
applied the margin definition (returned orders excluded, unit cost from the cost table) and grouped by category.

Now a harder one: **refund rate by sales region**. Refunds live in their own fact table, an order can be refunded
in installments, and the region is three joins away. Joining refunds to orders and dividing (what most people and
most LLMs write) counts an order's amount once per refund installment:

In [ ]:
naive = lu.trino("""
SELECT g.sales_region, SUM(f.amount) / SUM(o.amount) AS refund_rate
FROM iceberg.dbt_demo.fct_orders o
LEFT JOIN iceberg.dbt_demo.fct_refunds f ON f.order_id = o.order_id
JOIN iceberg.dbt_demo.stg_customers c ON c.customer_id = o.customer_id
JOIN iceberg.dbt_demo.dim_regions g ON g.country = c.country
GROUP BY 1 ORDER BY 1""")
answer = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["refund_rate"], "dimensions": ["dim_regions.sales_region"]})
semantic = pd.DataFrame(answer["rows"], columns=["sales_region", "refund_rate"]).sort_values("sales_region")
print("The semantic server aggregates refunds and sales separately (fan-out safe), then divides:\n")
print(answer["sql"], "\n")
naive.merge(semantic, on="sales_region", suffixes=(" (naive join)", " (semantic layer)"))

## 4. The LLM, step by step

Same idea, in plain English. `lu.ask_semantic` gives the LLM the server's MCP tools (as Ollama tool schemas) and a
short system prompt: *use the tools, never write SQL, never invent numbers*. Each turn the LLM either calls a tool,
which `lab_utils` runs against the server and hands back, or answers. Every step is recorded:

In [ ]:
QUESTION = "What was our gross margin per product category, and how does it compare with revenue?"
r = lu.ask_semantic(QUESTION)
print("Question:", QUESTION, "\n")
for i, step in enumerate(r["steps"], 1):
    print(f"── step {i}: the LLM calls {step['tool']}")
    print("   arguments:", json.dumps(step["args"]))
    res = step["result"]
    if "error" in res:
        print("   server says:", res["error"])
    elif "rows" in res:
        print("   server ran:", " ".join(res["sql"].split())[:300], "...")
        print("   rows:", res["rows"])
    else:
        print("   server returns:", json.dumps(res, default=str)[:300], "...")
print("\n── answer:", r["answer"])

The LLM discovered the metrics, chose `gross_margin` (and `total_revenue`) by `dim_products.category`, and the server
did the rest. The numbers in its answer come from the rows above, not from the model's imagination. The same
question **without** the semantic layer: the LLM gets the seven tables' DDL and writes Trino SQL itself.

In [ ]:
raw = lu.ask_raw_sql(QUESTION, TABLES)
print("SQL written by the LLM:\n", raw["sql"], "\n")
print("error:", raw["error"]) if raw["error"] else print("rows:", raw["rows"])

## 5. Six questions, both paths, scored

Ground truth is hand-written SQL that applies the certified definitions. Each question has a trap: the returns rule,
the cost table two joins away, refunds in installments (fan-out), a ratio, payments split in parts, a time grain.
Small local models vary between runs; the score table is the point, not one answer.

In [ ]:
REV = "SUM(CASE WHEN o.status <> 'returned' THEN o.amount ELSE 0 END)"
FROM = ("FROM iceberg.dbt_demo.fct_orders o "
        "JOIN iceberg.dbt_demo.stg_customers c ON c.customer_id = o.customer_id "
        "JOIN iceberg.dbt_demo.dim_regions g ON g.country = c.country "
        "JOIN iceberg.dbt_demo.dim_products p ON p.product_id = o.product_id "
        "JOIN iceberg.dbt_demo.dim_product_costs k ON k.product_id = o.product_id")
REFUNDS_BY_REGION = ("SELECT g.sales_region, SUM(f.amount) AS v FROM iceberg.dbt_demo.fct_refunds f "
                     "JOIN iceberg.dbt_demo.fct_orders o ON o.order_id = f.order_id "
                     "JOIN iceberg.dbt_demo.stg_customers c ON c.customer_id = o.customer_id "
                     "JOIN iceberg.dbt_demo.dim_regions g ON g.country = c.country GROUP BY 1")
QUESTIONS = {
    "What is total revenue by country?": f"SELECT c.country, {REV} {FROM} GROUP BY 1",
    "What is the gross margin per product category?":
        f"SELECT p.category, SUM(CASE WHEN o.status <> 'returned' THEN o.amount - o.quantity * k.unit_cost ELSE 0 END) {FROM} GROUP BY 1",
    "What is the refund rate in each sales region?":
        f"SELECT r.sales_region, r.v / s.v FROM ({REFUNDS_BY_REGION}) r "
        f"JOIN (SELECT g.sales_region, SUM(o.amount) AS v {FROM} GROUP BY 1) s ON s.sales_region = r.sales_region",
    "What is the average order value for each subscription plan?":
        f"SELECT c.plan, {REV} / COUNT(DISTINCT o.order_id) {FROM} GROUP BY 1",
    "How much money did we collect per payment method?":
        "SELECT method, SUM(amount) FROM iceberg.dbt_demo.fct_payments GROUP BY 1",
    "What was the revenue per week?": f"SELECT date_trunc('week', o.order_ts), {REV} {FROM} GROUP BY 1",
}

results = []
for q, truth_sql in QUESTIONS.items():
    truth = lu.trino(truth_sql).values.tolist()
    sem, raw = lu.ask_semantic(q), lu.ask_raw_sql(q, TABLES)
    results.append({"question": q,
                    "semantic": "✓" if lu.rows_match(sem["rows"], truth) else "✗",
                    "raw SQL": "✓" if lu.rows_match(raw["rows"], truth) else ("error" if raw["error"] else "✗"),
                    "tool calls": " → ".join(sem["tool_calls"]),
                    "semantic SQL": sem["sql"], "raw SQL text": raw["sql"]})
score = pd.DataFrame(results)
print(f"semantic layer: {(score['semantic'] == '✓').sum()}/{len(score)}   raw text-to-SQL: {(score['raw SQL'] == '✓').sum()}/{len(score)}")
score[["question", "semantic", "raw SQL", "tool calls"]]

In [ ]:
# The SQL behind each answer: look for missing returns rules, unit_price instead of unit_cost, and refunds joined to orders
for row in results:
    print("Q:", row["question"])
    print("  semantic:", " ".join((row["semantic SQL"] or "-").split()))
    print("  raw     :", " ".join((row["raw SQL text"] or "-").split()), "\n")

## 6. Governance

Roles come from the caller's identity (here the `X-Semantic-Role` header; on a shared cluster a JWT). The
`marketing` role may filter by country but never touch e-mail addresses; the server refuses instead of silently
dropping the filter:

In [ ]:
denied = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["order_count"],
                                      "filters": [{"field": "stg_customers.email", "op": "LIKE", "value": "%@example.com"}]},
                     role="marketing")
print("marketing, filtering on e-mail ->", denied.get("error") or denied)

`dach_sales` asks for revenue *without* any filter, and gets only its region: the server adds the role's row filter
to the SQL. Ask the LLM as that role and it cannot leak other regions either:

In [ ]:
everyone = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["total_revenue", "order_count"]})
dach = lu.mcp_call("query_metric", {"model": MODEL, "metrics": ["total_revenue", "order_count"]}, role="dach_sales")
print("SQL for dach_sales (note the sales_region predicate):\n", dach["sql"], "\n")
display(pd.DataFrame([everyone["rows"][0], dach["rows"][0]], columns=everyone["columns"], index=["analyst", "dach_sales"]))
print("LLM as dach_sales:", lu.ask_semantic("What is our total revenue?", role="dach_sales")["answer"])

## What to take away, and the limits

- The LLM never wrote SQL. It chose among **certified metrics**, so "revenue" means the same thing in every answer,
  dashboard and agent, and every answer comes with the SQL that produced it.
- The server knows the **joins and keys**: multi-hop joins (costs, regions) and fan-out across fact tables (refund
  installments) are handled once, in the model, not in every prompt.
- **Governance** is enforced by the server, per role, including row-level filters.
- Limits of this version (v0.1.1): fan-out protection covers ratios and `COUNT(DISTINCT ...)`. A plain sum of an
  order metric grouped by a *payment* dimension (e.g. revenue by payment method) would still count split orders
  twice; the model's instructions steer the LLM away from that combination. Ossie has no `TRINO` dialect yet, so the
  model uses `ANSI_SQL`. The `header` identity mode is only safe behind the chart's NetworkPolicy.